In [1]:
import cv2
import numpy as np
import datetime

In [2]:
def aruco_display(corners, ids, rejected, frame):
    image = frame.copy()
    ids = ids.flatten()
    
    for (markerCorner, markerID) in zip(corners, ids):
            
            corners = markerCorner.reshape((4, 2))
            (topLeft, topRight, bottomRight, bottomLeft) = corners
            
            topRight = (int(topRight[0]), int(topRight[1]))
            bottomRight = (int(bottomRight[0]), int(bottomRight[1]))
            bottomLeft = (int(bottomLeft[0]), int(bottomLeft[1]))
            topLeft = (int(topLeft[0]), int(topLeft[1]))

            cv2.line(image, topLeft, topRight, (0, 255, 0), 2)
            cv2.line(image, topRight, bottomRight, (0, 255, 0), 2)
            cv2.line(image, bottomRight, bottomLeft, (0, 255, 0), 2)
            cv2.line(image, bottomLeft, topLeft, (0, 255, 0), 2)
            
            cX = int((topLeft[0] + bottomRight[0]) / 2.0)
            cY = int((topLeft[1] + bottomRight[1]) / 2.0)
            cv2.circle(image, (cX, cY), 4, (0, 0, 255), -1)
            
            cv2.putText(image, str(markerID),(topLeft[0], topLeft[1] - 10), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (0, 255, 0), 2)
            # print("[Inference] ArUco marker ID: {}".format(markerID))
            
    return image

In [3]:
src_images = ['Images/Augment/01-sample.png',
              'Images/Augment/02-Resistor.png',
             'Images/Augment/03-Battery.png']

In [56]:
def image_aug(frame, corners, ids, x1,y1,x2,y2,x3,y3,x4,y4):
    h, w = frame.shape[:2]
    mask = np.zeros([h, w], dtype=np.uint8)

    warp_image = np.zeros_like(frame)

    for (markerCorner, markerID) in zip(corners, ids):
        try:
            src_image = cv2.imread(src_images[int(markerID)])
            src_h, src_w = src_image.shape[:2]
            # TopL, TopR, BottomR, BottomL
            src_pts = np.array([[x1, y1], [src_w+x2, y2], [src_w+x3, src_h+y3], [x4, src_h+y4]])

            crns = markerCorner.reshape([4, 2]).astype(int)
            
            cv2.fillConvexPoly(mask, crns, 255)
            frame = cv2.bitwise_and(frame, frame, mask=cv2.bitwise_not(mask))

            H, _ = cv2.findHomography(srcPoints=src_pts, dstPoints=crns)
            augmented_image = cv2.warpPerspective(src_image, H, (w, h))
            
            warp_image = cv2.bitwise_or(warp_image, augmented_image)
        except IndexError as e:
            print(f"Error: {e}, marker ID: {int(markerID)}")
            

    # Combine the original frame with the augmented warp_image
    final_image = cv2.bitwise_or(frame, warp_image)

    return final_image

In [57]:
# Parameters
arucoDict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
arucoParams = cv2.aruco.DetectorParameters()

In [61]:
cap = cv2.VideoCapture(0)

def update_src_pts(x):
    pass

cv2.namedWindow('Parameters')
cv2.resizeWindow('Parameters', 400, 400)
cv2.createTrackbar('x1', 'Parameters', 100, 200, update_src_pts)
cv2.createTrackbar('y1', 'Parameters', 100, 200, update_src_pts)
cv2.createTrackbar('x2', 'Parameters', 100, 200, update_src_pts)
cv2.createTrackbar('y2', 'Parameters', 100, 200, update_src_pts)
cv2.createTrackbar('x3', 'Parameters', 100, 200, update_src_pts)
cv2.createTrackbar('y3', 'Parameters', 100, 200, update_src_pts)
cv2.createTrackbar('x4', 'Parameters', 100, 200, update_src_pts)
cv2.createTrackbar('y4', 'Parameters', 100, 200, update_src_pts)

while cap.isOpened():
    
    _, img = cap.read()
    
    corners, ids, rejected = cv2.aruco.detectMarkers(img, arucoDict, parameters=arucoParams)
    
    x1 = cv2.getTrackbarPos('x1', 'Parameters')
    y1 = cv2.getTrackbarPos('y1', 'Parameters')
    x2 = cv2.getTrackbarPos('x2', 'Parameters')
    y2 = cv2.getTrackbarPos('y2', 'Parameters')
    x3 = cv2.getTrackbarPos('x3', 'Parameters')
    y3 = cv2.getTrackbarPos('y3', 'Parameters')
    x4 = cv2.getTrackbarPos('x4', 'Parameters')
    y4 = cv2.getTrackbarPos('y4', 'Parameters')
    
    if len(corners) > 0:
        detected_markers = aruco_display(corners, ids, rejected, img)
        wimage = image_aug(img, corners, ids, 100-x1,100-y1,100-x2,100-y2,100-x3,100-y3,100-x4,100-y4)
            
        cv2.imshow("Detected Markers", detected_markers)
        cv2.imshow("Wrap Image", wimage)
        
    cv2.imshow("Image", img)
    
    if cv2.waitKey(25) & 0xFF == 27:
        break
        
cap.release()
cv2.destroyAllWindows()

In [59]:
cap.release()
cv2.destroyAllWindows()